# Trotter robustness — spectral-gap scaling

**Normalized Ising instances · $\beta=4$ · $n=3,\ldots,10$ · 100 disorder instances per system size**

Publication-style plots following the supplied `plot_main_runtime.py`, `plot_main_queries.py`, and `plot_aux_updated_4.py` conventions: APS-sized figures, APS-style categorical palette with differentiated markers and line styles, light horizontal grids, frameless legends, and median/IQR statistics.

- **Fixed $(\gamma,t)$ discretization:** compare all ten Trotter repetition counts.
- **Fixed Trotter repetition count:** compare the four $2^k\times2^k$ parameter grids.

All plot styling and export preferences are configured in the first code cell.

The notebook only reads `spectral_gaps_trotter_merged.hdf5`; run `python spectral_gaps_trotter.py collide` after computation.

In [12]:
from pathlib import Path
import h5py
import matplotlib as mpl
import numpy as np

# Select the original backend before importing pyplot.
mpl.use("pgf")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import LogFormatterMathtext, LogLocator, NullFormatter


# All plot styling and export preferences are defined in this cell.
# The original first-cell preferences take priority over later settings.
REGULAR_SIZE = 10  # Axis labels, titles, and the global fallback size
SMALL_SIZE = 8     # Tick labels and legends
CM = 1.0 / 2.54
APS_FIGURESTAR_WIDTH_CM = 17.8

SUMMARY = "median"                # "median" or "mean"
SHOW_IQR_R = True                # Error bars when varying Trotter repetitions
SHOW_IQR_K = True                # Error bars when varying the parameter grid
SAVE_FIGURES = True
OUTPUT_DIR = Path("plots")
EXPORT_FORMATS = ("pdf", "png", "pgf")
EXPORT_BBOX = "tight"

mpl.rcParams.update({
    # Original PGF, LaTeX, and serif-font preferences.
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "font.family": "serif",
    "pgf.preamble": "",
    "font.size": REGULAR_SIZE,
    "axes.labelsize": REGULAR_SIZE,
    "axes.titlesize": REGULAR_SIZE,
    "xtick.labelsize": SMALL_SIZE,
    "ytick.labelsize": SMALL_SIZE,
    "legend.fontsize": SMALL_SIZE,
    "figure.titlesize": REGULAR_SIZE,
    # Nonconflicting appearance settings moved from the later style block.
    "axes.linewidth": 0.7,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.grid.which": "major",
    "grid.color": "0.92",
    "grid.linewidth": 0.55,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3.0,
    "ytick.major.size": 3.0,
    "legend.frameon": False,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
})

# Colors, markers, and line styles for each Trotter repetition count.
STEP_VALUES = (1, 2, 5, 10, 20, 30, 40, 50, 75, 100)
STEP_COLORS = [
    "#B7D0E5", "#85B0D1", "#528FB8", "#246D9E", "#004488",
    "#EBC9DF", "#DDA3C5", "#CD7DAB", "#BA568F", "#AA3377",
]
STEP_MARKERS = ["o", "s", "^", "D", "v", "o", "s", "^", "D", "v"]
R_COLORS = dict(zip(STEP_VALUES, STEP_COLORS))
R_MARKERS = dict(zip(STEP_VALUES, STEP_MARKERS))
R_STYLES = {r: ("-" if i < 5 else "--") for i, r in enumerate(STEP_VALUES)}
GRID_COLORS = ["#222222", "#004488", "#EE7733", "#AA3377"]
K_COLORS = dict(zip((1, 2, 3, 4), GRID_COLORS))

ERRORBAR_STYLES = {
    "compact": dict(markersize=2.6, capsize=1.3, linewidth=1.05,
                    elinewidth=0.58, capthick=0.55, alpha=0.84, zorder=3),
    "regular": dict(markersize=2.7, capsize=1.8, linewidth=1.15,
                    elinewidth=0.8, capthick=0.75, alpha=0.94, zorder=3),
}
LINE_STYLE = dict(markersize=2.7, linewidth=1.25, alpha=0.94, zorder=3)
X_MARGIN = 0.2
LOG_MINOR_SUBS = np.arange(2, 10) * 0.1

# Figure-specific sizes, title padding, legend styling, and panel spacing.
FIGURE_SETTINGS = {
    "fixed_grid": {
        "subplots": dict(nrows=2, ncols=2,
                         figsize=(APS_FIGURESTAR_WIDTH_CM * CM, 12.0 * CM),
                         sharex=True, sharey=True),
        "title_pad": 5,
        "xticks": None,  # Use all system sizes from the loaded data.
        "x_offset_step": 0.028,
        "legend_handles": dict(linewidth=1.15, markersize=3.2),
        "legend": dict(loc="lower center", bbox_to_anchor=(0.5, 0.015),
                       ncol=5, columnspacing=1.2, handlelength=1.7,
                       labelspacing=0.6),
        "spacing": dict(left=0.115, right=0.985, top=0.94, bottom=0.19,
                        wspace=0.22, hspace=0.25),
    },
    "fixed_steps": {
        "subplots": dict(nrows=2, ncols=5,
                         figsize=(APS_FIGURESTAR_WIDTH_CM * CM, 9.6 * CM),
                         sharex=True, sharey=True),
        "title_pad": 4,
        "xticks": [3, 6, 10],
        "legend_handles": dict(linewidth=1.35, markersize=3.0,
                               marker="o", linestyle="-"),
        "legend": dict(loc="lower center", bbox_to_anchor=(0.5, 0.02),
                       ncol=4, columnspacing=1.5, handlelength=1.8),
        "spacing": dict(left=0.095, right=0.99, top=0.94, bottom=0.18,
                        wspace=0.24, hspace=0.32),
    },
}


def style_axis(ax, ticks=None):
    ax.set_yscale("log")
    ax.set_xlim(float(NS[0]) - X_MARGIN, float(NS[-1]) + X_MARGIN)
    ax.set_xticks(NS if ticks is None else ticks)
    ax.yaxis.set_major_formatter(LogFormatterMathtext())
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=LOG_MINOR_SUBS))
    ax.yaxis.set_minor_formatter(NullFormatter())


def draw_series(ax, values, color, show_iqr, label=None, *, marker="o", linestyle="-", x_offset=0.0, compact=False):
    # summarize() and NS are supplied by the subsequent data/statistics cells.
    center, lo, hi = summarize(values)
    good = np.isfinite(center) & (center > 0)
    if not good.any():
        return
    x = NS[good] + x_offset
    if show_iqr:
        yerr = [np.maximum(center[good] - lo[good], 0),
                np.maximum(hi[good] - center[good], 0)]
        style = ERRORBAR_STYLES["compact" if compact else "regular"]
        ax.errorbar(x, center[good], yerr=yerr, fmt=marker,
                    linestyle=linestyle, color=color, label=label, **style)
    else:
        ax.plot(x, center[good], color=color, linestyle=linestyle,
                marker=marker, label=label, **LINE_STYLE)


def save_figure(fig, basename):
    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        for ext in EXPORT_FORMATS:
            fig.savefig(OUTPUT_DIR / f"{basename}.{ext}", bbox_inches=EXPORT_BBOX)


## 1. Read the merged spectral gaps

In [13]:
FILENAME = "spectral_gaps_trotter_merged.hdf5"
CANDIDATES = [Path.cwd() / FILENAME, Path.cwd() / "trotter_robustness" / FILENAME]
FILE = next((p for p in CANDIDATES if p.is_file()), None)
if FILE is None:
    raise FileNotFoundError(f"Cannot find {FILENAME}. Run 'python spectral_gaps_trotter.py collide' first.")

NS = np.arange(3, 11)
COUNT = 100
with h5py.File(FILE, "r") as f:
    RS = np.asarray(f.attrs["rs"], dtype=int)
    KS = np.asarray(f.attrs["ks"], dtype=int)
    BETA = float(f.attrs["beta"])
    if not np.isclose(BETA, 4.0):
        raise ValueError(f"Expected beta=4, found {BETA}.")
    # [discretization k, Trotter steps r, system size n, instance idx]
    GAPS = np.full((len(KS), len(RS), len(NS), COUNT), np.nan)
    for ki, k in enumerate(KS):
        for ni, n in enumerate(NS):
            for idx in range(COUNT):
                key = f"k={k}/{n}/{idx}"
                if key in f:
                    values = np.asarray(f[key][:], dtype=float)
                    if values.shape != (len(RS),):
                        raise ValueError(f"Invalid shape {values.shape} at {key}")
                    GAPS[ki, :, ni, idx] = values

print(f"Source: {FILE.resolve()}")
print(f"beta={BETA:g}; n={list(NS)}; r={list(RS)}; k={list(KS)}")
print(f"Available: {np.isfinite(GAPS).sum():,} / {GAPS.size:,} gap values")

Source: /mnt/c/Users/massi/Documents/progetti/pubmonaqa/robustness_trotterization/spectral_gaps_trotter_merged.hdf5
beta=4; n=[np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]; r=[np.int64(1), np.int64(2), np.int64(5), np.int64(10), np.int64(20), np.int64(30), np.int64(40), np.int64(50), np.int64(75), np.int64(100)]; k=[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Available: 32,000 / 32,000 gap values


## 2. Summary statistics

Plot styling, layout, and export preferences are defined in the first code cell.


In [14]:
# Nonpositive or missing spectral gaps cannot be shown on a logarithmic axis.
VALID = np.where(np.isfinite(GAPS) & (GAPS > 0.0), GAPS, np.nan)

def summarize(data):
    """Return center and error-bar bounds for [n, instance]."""
    if SUMMARY not in {"mean", "median"}:
        raise ValueError("SUMMARY must be 'median' or 'mean'.")
    result = np.full((3, len(NS)), np.nan)
    for ni, row in enumerate(data):
        row = row[np.isfinite(row)]
        if row.size:
            if SUMMARY == "median":
                result[0, ni] = np.median(row)
                result[1:, ni] = np.percentile(row, [25, 75])
            else:
                mean, std = np.mean(row), np.std(row)
                result[:, ni] = mean, max(mean - std, np.finfo(float).tiny), mean + std
    return result


## 3. Fixed discretization — varying Trotter repetitions

Each panel shows **median spectral gaps with 25th–75th percentile error bars** for all ten Trotter repetition counts. Categorical colors, differentiated markers, and solid/dashed line styles improve identification of overlapping curves. A small horizontal offset (at most 0.13 in $n$) separates the error bars visually; the actual system sizes remain the integers on the axis.

In [15]:
settings = FIGURE_SETTINGS["fixed_grid"]
fig, axes = plt.subplots(**settings["subplots"])

for ki, (k, ax) in enumerate(zip(KS, axes.flat)):
    for ri, r in enumerate(RS):
        # Offset the error bars without changing the integer system-size ticks.
        offset = (ri - (len(RS) - 1) / 2) * settings["x_offset_step"]
        draw_series(ax, VALID[ki, ri], R_COLORS[int(r)], SHOW_IQR_R,
                    marker=R_MARKERS[int(r)], linestyle=R_STYLES[int(r)],
                    x_offset=offset, compact=True)
    style_axis(ax, ticks=settings["xticks"])
    ax.set_title(rf"({chr(97 + ki)}) $2^{int(k)}\times 2^{int(k)}$ grid",
                 pad=settings["title_pad"])

for ax in axes[-1, :]:
    ax.set_xlabel(r"$n$")
for ax in axes[:, 0]:
    ax.set_ylabel(r"Spectral gap $\delta$")

handles = [Line2D([0], [0], color=R_COLORS[int(r)],
                  marker=R_MARKERS[int(r)], linestyle=R_STYLES[int(r)],
                  label=rf"$r={int(r)}$", **settings["legend_handles"]) for r in RS]
fig.legend(handles=handles, **settings["legend"])
fig.subplots_adjust(**settings["spacing"])
save_figure(fig, "trotter_scaling_fixed_grid")
plt.show()


/tmp/ipykernel_14590/733932561.py:26: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


## 4. Fixed Trotter repetitions — varying discretization

Each panel fixes $r$ and compares the $2\times2$, $4\times4$, $8\times8$, and $16\times16$ midpoint grids. These plots use the median and IQR by default, consistent with the supplied auxiliary figures. All panels use the same logarithmic vertical scale.

In [16]:
settings = FIGURE_SETTINGS["fixed_steps"]
fig, axes = plt.subplots(**settings["subplots"])

for ri, (r, ax) in enumerate(zip(RS, axes.flat)):
    for ki, k in enumerate(KS):
        draw_series(ax, VALID[ki, ri], K_COLORS[int(k)], SHOW_IQR_K)
    style_axis(ax, ticks=settings["xticks"])
    ax.set_title(rf"({chr(97 + ri)}) $r={int(r)}$", pad=settings["title_pad"])

for ax in axes[-1, :]:
    ax.set_xlabel(r"$n$")
for ax in axes[:, 0]:
    ax.set_ylabel(r"Spectral gap $\delta$")

handles = [Line2D([0], [0], color=K_COLORS[int(k)],
                  label=rf"$2^{int(k)}\times 2^{int(k)}$",
                  **settings["legend_handles"]) for k in KS]
fig.legend(handles=handles, **settings["legend"])
fig.subplots_adjust(**settings["spacing"])
save_figure(fig, "trotter_scaling_fixed_steps")
plt.show()


/tmp/ipykernel_14590/4110144598.py:21: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


## 5. Data coverage

In [6]:
counts = np.isfinite(GAPS).sum(axis=-1)  # [k, r, n]
incomplete = np.argwhere(counts < COUNT)
if not len(incomplete):
    print(f"All {GAPS.size:,} results are present.")
else:
    print(f"Incomplete combinations: {len(incomplete)} / {counts.size}")
    print(f"{'grid':>7} {'r':>5} {'n':>4} {'count':>8}")
    for ki, ri, ni in incomplete[:40]:
        size = 2 ** int(KS[ki])
        print(f"{size:>2}x{size:<2} {RS[ri]:>5} {NS[ni]:>4} {counts[ki, ri, ni]:>5}/{COUNT}")
    if len(incomplete) > 40:
        print(f"... and {len(incomplete) - 40} additional incomplete combinations")

All 32,000 results are present.
